# fact_invoice — Star Schema Fact Loader (PySpark)

In [ ]:
# ============================================================
#  fact_invoice  — Star schema fact loader (PySpark)
# ============================================================
from pyspark.sql import functions as F

dbutils.widgets.dropdown(name="environment", defaultValue="dev",
                         choices=["dev","qa","prd"], label="select Environment")
env = dbutils.widgets.get("environment")

silver_inv = f"saleslake_{env}.silver_{env}.cleanedInvoice"
gold_fact  = f"saleslake_{env}.gold_{env}.fact_invoice"
gold_g     = f"saleslake_{env}.gold_{env}"

# Watermark
watermark = (spark.sql(
    f"SELECT COALESCE(MAX(load_ts), TO_TIMESTAMP('1990-01-01')) AS w "
    f"FROM {gold_fact}").first()["w"])

df_i = spark.table(silver_inv).filter(F.col("ingest_ts") > F.lit(watermark))

d_cust = spark.table(f"{gold_g}.refinedCustomer").filter("is_active = 'Y'") \
              .select("customer_sk", "customer_id")
d_stor = spark.table(f"{gold_g}.dim_store").filter("is_active = 'Y'") \
              .select("store_sk", "store_id", "region_id")
d_reg  = spark.table(f"{gold_g}.dim_region").select("region_sk", "region_id", "region_name")
d_disc = spark.table(f"{gold_g}.dim_discount").filter("is_active = 'Y'") \
              .select("discount_sk", "discount_code")
d_date = spark.table(f"{gold_g}.dim_date").select(
            F.col("date_sk"), F.col("full_date"))

# Bring in store -> region link
fact = (df_i
        .withColumn("store_id_int", F.col("store_id").cast("int"))
        .join(d_cust, "customer_id", "left")
        .join(d_stor, df_i["store_id"].cast("int") == d_stor["store_id"], "left")
        .join(d_reg.alias("r"),
              (F.col("r.region_name") == F.upper(F.col("region"))), "left")
        .join(d_disc, "discount_code", "left")
        .join(d_date.alias("dinv"),  F.col("invoice_date") == F.col("dinv.full_date"), "left")
        .join(d_date.alias("ddue"),  F.col("due_date")     == F.col("ddue.full_date"), "left")
        .join(d_date.alias("dpay"),  F.col("payment_date") == F.col("dpay.full_date"), "left")
        .withColumn("days_to_pay",
              F.datediff(F.col("payment_date"), F.col("invoice_date")))
        .withColumn("is_overdue",
              (F.col("payment_status").isin("OVERDUE")) |
              ((F.col("payment_status").isin("PENDING")) &
               (F.current_date() > F.col("due_date"))))
        .select(
            F.col("invoice_id"), F.col("invoice_number"),
            F.col("customer_sk"), F.col("store_sk"),
            F.col("region_sk"),   F.col("discount_sk"),
            F.col("dinv.date_sk").alias("invoice_date_sk"),
            F.col("ddue.date_sk").alias("due_date_sk"),
            F.col("dpay.date_sk").alias("payment_date_sk"),
            F.col("invoice_date"), F.col("due_date"), F.col("payment_date"),
            F.col("subtotal_amount"), F.col("discount_amount"),
            F.col("tax_amount"), F.col("total_amount"),
            F.col("payment_status"), F.col("payment_method"), F.col("channel"),
            F.col("days_to_pay"), F.col("is_overdue"),
            F.current_timestamp().alias("load_ts"),
        ))

fact.write.format("delta").mode("append").saveAsTable(gold_fact)
print(f"fact_invoice loaded: +{fact.count()} rows")
